# Przygotowanie danych – uniwersalny pipeline (13 wskaźników)

## 1. Konfiguracja

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

WSKAZNIKI = ["X1", "X2", "X3", "X5", "X6", "X24", "X29", "X34", "X35", "X46", "X56", "X58", "X62"]
PLIKI = []
KATALOG_WEJSCIOWY = Path(".")
OBSLUGIWANE_ROZSZERZENIA = {".arff", ".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet"}

KATALOG_WYJSCIOWY = Path("gotowe dane")
ZAPISZ_ARFF = True
ZAPISZ_EXCEL = True
ZAPISZ_CSV = False    
SUFIKS_NAZWY = "_13wskaznikow"   
NAZWA_KOLUMNY_ETYKIETY_W_ARFF = "class"   
WYMAGAJ_WSZYSTKICH_WSKAZNIKOW = True

NAZWY_ETYKIETY = ["class", "y", "target", "label", "bankrupt", "bankruptcy",
                  "status_label", "status", "klasa"]
ETYKIETA_1 = {"1", "1.0", "true", "yes", "tak", "failed", "bankrupt", "bankrut", "default"}   # bankructwo
ETYKIETA_0 = {"0", "0.0", "false", "no", "nie", "alive", "active", "healthy"}                  # brak bankructwa

## 2. Funkcje (każda podejmuje decyzję na podstawie zawartości pliku)
- **format** – wybierany po rozszerzeniu (CSV: separator wykrywany automatycznie, kodowanie UTF‑8 → CP1250 → Latin‑1),
- **nazwy kolumn** – `Attr12`, `attr12`, `x12`, `A12` → `X12`,
- **etykieta** – znajdowana po nazwie, a gdy jej nie ma: ostatnia kolumna dwuwartościowa; wartości tekstowe (`failed`/`alive`, `b'1'` itd.) są zamieniane na 0/1,
- **liczby** – teksty typu `"1,25"` lub `"?"` są zamieniane na liczby / `NaN`, `±inf` → `NaN`,
- **duplikaty** – usuwane przed wyborem 13 wskaźników (na pełnym zestawie cech, żeby nie usuwać firm różniących się pozostałymi wskaźnikami); przy sprzecznych etykietach zostaje wiersz z bankructwem.

In [2]:
RE_WSKAZNIK = re.compile(r"^(?:attr|x|a)?[\s_\-]*(\d+)$", re.IGNORECASE)
RE_X = re.compile(r"X\d+")
BRAKI_TEKST = {"?", "", "nan", "na", "n/a", "null", "none", "-"}


def wczytaj(sciezka, log):
    ext = sciezka.suffix.lower()
    if ext == ".arff":
        from scipy.io import arff
        rek, _ = arff.loadarff(sciezka)
        df = pd.DataFrame(rek)
        for c in df.columns:                      # bajty (b'0') -> tekst
            if df[c].dtype == object:
                df[c] = df[c].map(lambda v: v.decode("utf-8") if isinstance(v, bytes) else v)
        log.append("Format: ARFF")
    elif ext in {".csv", ".tsv", ".txt"}:
        for kodowanie in ("utf-8", "cp1250", "latin-1"):
            try:
                df = pd.read_csv(sciezka, sep=None, engine="python", encoding=kodowanie)
                log.append(f"Format: tekstowy (separator wykryty automatycznie, kodowanie {kodowanie})")
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError("Nie udało się odczytać pliku w żadnym z kodowań.")
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(sciezka)
        log.append("Format: Excel (pierwszy arkusz)")
    elif ext == ".parquet":
        df = pd.read_parquet(sciezka)
        log.append("Format: Parquet")
    else:
        raise ValueError(f"Nieobsługiwane rozszerzenie: {ext}")
    return df


def ujednolic_nazwy(df, log):
    zmiany = {c: f"X{int(m.group(1))}" for c in df.columns if (m := RE_WSKAZNIK.match(str(c).strip()))}
    if len(set(zmiany.values())) < len(zmiany):
        raise ValueError("Po ujednoliceniu nazw powstałyby duplikaty kolumn.")
    przyklad = next(((a, b) for a, b in zmiany.items() if str(a) != b), None)
    if przyklad:
        log.append(f"Ujednolicono nazwy {len(zmiany)} kolumn wskaźników (np. {przyklad[0]} -> {przyklad[1]})")
    return df.rename(columns=zmiany)


def znajdz_etykiete(df, log):
    kandydaci = [c for c in df.columns if not RE_X.fullmatch(str(c))]
    po_nazwie = {str(c).strip().lower(): c for c in kandydaci}
    for n in NAZWY_ETYKIETY:
        if n in po_nazwie:
            log.append(f"Etykieta: kolumna '{po_nazwie[n]}' (rozpoznana po nazwie)")
            return po_nazwie[n]
    dwuwartosciowe = [c for c in kandydaci if 1 <= df[c].dropna().nunique() <= 2]
    if dwuwartosciowe:
        log.append(f"Etykieta: kolumna '{dwuwartosciowe[-1]}' (nie znaleziono po nazwie – wzięto ostatnią kolumnę dwuwartościową)")
        return dwuwartosciowe[-1]
    raise ValueError("Nie znaleziono kolumny z etykietą (bankructwo / brak bankructwa).")


def koduj_etykiete(s):
    tekst = s.astype(str).str.strip().str.lower()
    tekst = tekst.mask(s.isna() | tekst.isin(BRAKI_TEKST))
    wynik = tekst.map(lambda v: np.nan if pd.isna(v) else 1 if v in ETYKIETA_1 else 0 if v in ETYKIETA_0 else "?")
    nieznane = sorted(set(tekst[wynik == "?"].dropna()))
    if nieznane:
        raise ValueError(f"Nierozpoznane wartości etykiety: {nieznane[:5]} – uzupełnij ETYKIETA_0 / ETYKIETA_1.")
    return wynik


def na_liczby(df, kolumny, log, ostrzezenia):
    out = {}
    for c in kolumny:
        s = df[c]
        if not pd.api.types.is_numeric_dtype(s):
            tekst = s.astype(str).str.strip().str.replace("\u00a0", "", regex=False).str.replace(",", ".", regex=False)
            tekst = tekst.mask(s.isna() | tekst.str.lower().isin(BRAKI_TEKST))
            przed = int(tekst.isna().sum())
            s2 = pd.to_numeric(tekst, errors="coerce")
            nowe = int(s2.isna().sum()) - przed
            if nowe > 0:
                ostrzezenia.append(f"{c}: {nowe} wartości nie dało się zamienić na liczby -> NaN")
            s = s2
        out[c] = s.astype(float)
    return pd.DataFrame(out, index=df.index)


def zapisz_arff(X, y, sciezka, nazwa_relacji):
    """Zapisuje dane w formacie ARFF (braki jako '?'), czytelnym m.in. przez scipy.io.arff."""
    with open(sciezka, "w", encoding="utf-8") as f:
        f.write(f"@relation {nazwa_relacji}\n\n")
        for c in X.columns:
            f.write(f"@attribute {c} numeric\n")
        f.write(f"@attribute {NAZWA_KOLUMNY_ETYKIETY_W_ARFF} {{0,1}}\n\n@data\n")
        wartosci = X.to_numpy(dtype=float)
        for wiersz, etykieta in zip(wartosci, y.to_numpy()):
            f.write(",".join("?" if np.isnan(v) else repr(float(v)) for v in wiersz) + f",{int(etykieta)}\n")


def zapisz_wyniki(X, y, plik):
    """Zapisuje przygotowane dane do katalogu KATALOG_WYJSCIOWY i zwraca listę zapisanych plików."""
    KATALOG_WYJSCIOWY.mkdir(parents=True, exist_ok=True)
    baza = KATALOG_WYJSCIOWY / f"{plik.stem}{SUFIKS_NAZWY}"
    zapisane = []
    if ZAPISZ_ARFF:
        zapisz_arff(X, y, baza.with_suffix(".arff"), baza.name)
        zapisane.append(baza.with_suffix(".arff"))
    if ZAPISZ_EXCEL or ZAPISZ_CSV:
        calosc = pd.concat([X, y.rename(NAZWA_KOLUMNY_ETYKIETY_W_ARFF)], axis=1)
        if ZAPISZ_EXCEL:
            calosc.to_excel(baza.with_suffix(".xlsx"), index=False, sheet_name="dane")
            zapisane.append(baza.with_suffix(".xlsx"))
        if ZAPISZ_CSV:
            calosc.to_csv(baza.with_suffix(".csv"), index=False)
            zapisane.append(baza.with_suffix(".csv"))
    return zapisane

## 3. Główna funkcja `przygotuj_dane`

In [3]:
def przygotuj_dane(sciezka):
    """Zwraca (X, y, raport). X ma wyłącznie kolumny z WSKAZNIKI, braki (NaN) są nietknięte."""
    sciezka = Path(sciezka)
    log, ostrz = [], []

    # 1. wczytanie + nazwy + etykieta
    df = wczytaj(sciezka, log)
    n_wejscie = len(df)
    df = ujednolic_nazwy(df, log)
    kol_y = znajdz_etykiete(df, log)
    cechy_wszystkie = [c for c in df.columns if RE_X.fullmatch(str(c))]
    log.append(f"Wiersze: {n_wejscie}, wskaźników w pliku: {len(cechy_wszystkie)}")

    # 2. kontrola, czy są potrzebne wskaźniki
    brakujace = [c for c in WSKAZNIKI if c not in cechy_wszystkie]
    if brakujace and WYMAGAJ_WSZYSTKICH_WSKAZNIKOW:
        raise KeyError(f"W pliku brakuje wskaźników: {brakujace}")
    cechy = [c for c in WSKAZNIKI if c in cechy_wszystkie]
    if brakujace:
        ostrz.append(f"Brak wskaźników w pliku (pominięto): {brakujace}")

    # 3. etykieta -> 0/1, wiersze bez etykiety odpadają
    y = koduj_etykiete(df[kol_y])
    bez_etykiety = int(y.isna().sum())
    if bez_etykiety:
        log.append(f"Usunięto {bez_etykiety} wierszy bez etykiety")
    dane = pd.concat([na_liczby(df, cechy_wszystkie, log, ostrz), y.rename("y")], axis=1)
    dane = dane[dane["y"].notna()].copy()
    dane["y"] = dane["y"].astype(int)

    # 4. wartości nieskończone -> NaN
    n_inf = int(np.isinf(dane[cechy_wszystkie]).sum().sum())
    if n_inf:
        dane[cechy_wszystkie] = dane[cechy_wszystkie].replace([np.inf, -np.inf], np.nan)
        log.append(f"Zamieniono {n_inf} wartości ±inf na NaN")

    # 5. duplikaty (na pełnym zestawie cech); przy sprzecznej etykiecie zostaje bankructwo
    przed = len(dane)
    konflikty = int(dane.drop_duplicates().duplicated(subset=cechy_wszystkie).sum())
    dane = (dane.sort_values("y", ascending=False, kind="stable")
                .drop_duplicates(subset=cechy_wszystkie, keep="first")
                .sort_index())
    n_dup = przed - len(dane)
    if n_dup:
        log.append(f"Usunięto {n_dup} duplikatów" + (f" (w tym {konflikty} ze sprzeczną etykietą – zostawiono wiersz z bankructwem)" if konflikty else ""))
    if konflikty:
        ostrz.append(f"{konflikty} duplikatów cech miało różne etykiety")

    # 6. wybór 13 wskaźników + usunięcie wierszy bez żadnej informacji
    X = dane[cechy].copy()
    y = dane["y"].copy()
    puste = X.isna().all(axis=1)
    if puste.any():
        log.append(f"Usunięto {int(puste.sum())} wierszy, w których wszystkie wybrane wskaźniki są puste")
        X, y = X[~puste], y[~puste]
    X, y = X.reset_index(drop=True), y.reset_index(drop=True)

    # 7. kontrole końcowe (tylko ostrzeżenia – zestaw wskaźników jest narzucony)
    stale = [c for c in cechy if X[c].nunique() <= 1]
    if stale:
        ostrz.append(f"Kolumny stałe lub puste: {stale}")
    if y.nunique() < 2:
        ostrz.append("W danych jest tylko jedna klasa – modele nie będą mogły się uczyć")
    elif y.sum() < 20:
        ostrz.append(f"Bardzo mało bankructw ({int(y.sum())}) – wyniki walidacji będą niestabilne")
    log.append(f"Wynik: {X.shape[0]} wierszy x {X.shape[1]} wskaźników, bankructwa: {y.mean():.2%}, braki w danych: {X.isna().mean().mean():.1%}")

    raport = {"plik": sciezka.name, "wiersze_wejscie": n_wejscie, "wiersze_wyjscie": len(X),
              "bankructwa_%": round(100 * y.mean(), 2), "bez_etykiety": bez_etykiety,
              "inf": n_inf, "duplikaty": n_dup, "konflikty_etykiet": konflikty,
              "puste_wiersze": int(puste.sum()), "kolumna_etykiety": kol_y,
              "log": log, "ostrzezenia": ostrz}
    return X, y, raport

## 4. Uruchomienie na wszystkich plikach
Błąd w jednym pliku nie zatrzymuje pozostałych – zostaje zapisany w podsumowaniu.

In [4]:
if PLIKI:
    pliki = [Path(p) for p in PLIKI]
else:
    pliki = sorted(p for p in Path(KATALOG_WEJSCIOWY).iterdir()
                   if p.is_file() and p.suffix.lower() in OBSLUGIWANE_ROZSZERZENIA
                   and not p.stem.endswith(SUFIKS_NAZWY))
print(f"Pliki do przetworzenia ({len(pliki)}):", [p.name for p in pliki])

DANE, RAPORTY, BLEDY = {}, [], {}
for p in pliki:
    print("\n" + "=" * 70 + f"\n{p.name}")
    try:
        X, y, r = przygotuj_dane(p)
    except Exception as e:
        BLEDY[p.name] = f"{type(e).__name__}: {e}"
        print("  BŁĄD:", BLEDY[p.name])
        continue
    for linia in r["log"]:
        print("  -", linia)
    for o in r["ostrzezenia"]:
        print("  ! UWAGA:", o)
    DANE[p.stem] = (X, y)
    RAPORTY.append(r)
    for zapisany in zapisz_wyniki(X, y, p):
        print("  Zapisano:", zapisany)

Pliki do przetworzenia (5): ['3year.arff', '4year.arff', '5year.arff', 'xgboost_wagi_podsumowanie.csv', 'xgboost_wagi_wyniki_foldow.csv']

3year.arff
  - Format: ARFF
  - Ujednolicono nazwy 64 kolumn wskaźników (np. Attr1 -> X1)
  - Etykieta: kolumna 'class' (rozpoznana po nazwie)
  - Wiersze: 10503, wskaźników w pliku: 64
  - Usunięto 87 duplikatów
  - Wynik: 10416 wierszy x 13 wskaźników, bankructwa: 4.73%, braki w danych: 0.3%
  Zapisano: gotowe dane\3year_13wskaznikow.arff
  Zapisano: gotowe dane\3year_13wskaznikow.xlsx

4year.arff
  - Format: ARFF
  - Ujednolicono nazwy 64 kolumn wskaźników (np. Attr1 -> X1)
  - Etykieta: kolumna 'class' (rozpoznana po nazwie)
  - Wiersze: 9792, wskaźników w pliku: 64
  - Usunięto 82 duplikatów
  - Wynik: 9710 wierszy x 13 wskaźników, bankructwa: 5.28%, braki w danych: 0.3%
  Zapisano: gotowe dane\4year_13wskaznikow.arff
  Zapisano: gotowe dane\4year_13wskaznikow.xlsx

5year.arff
  - Format: ARFF
  - Ujednolicono nazwy 64 kolumn wskaźników (np. At

## 5. Podsumowanie

In [5]:
if RAPORTY:
    podsumowanie = pd.DataFrame(RAPORTY).drop(columns=["log", "ostrzezenia"]).set_index("plik")
    display(podsumowanie)
    print("Udział braków (%) w 13 wskaźnikach:")
    display(pd.DataFrame({n: X.isna().mean() * 100 for n, (X, _) in DANE.items()}).T.round(1))
if BLEDY:
    print("Pliki z błędem:")
    display(pd.Series(BLEDY, name="błąd").to_frame())

,wiersze_wejscie,wiersze_wyjscie,bankructwa_%,bez_etykiety,inf,duplikaty,konflikty_etykiet,puste_wiersze,kolumna_etykiety
plik,,,,,,,,,
3year.arff,10503,10416,4.73,0,0,87,0,0,class
4year.arff,9792,9710,5.28,0,0,82,0,0,class
5year.arff,5910,5850,6.97,0,0,60,0,0,class


Udział braków (%) w 13 wskaźnikach:


,X1,X2,X3,X5,X6,X24,X29,X34,X35,X46,X56,X58,X62
3year,0.0,0.0,0.0,0.2,0.0,2.2,0.0,0.1,0.0,0.2,0.4,0.3,0.4
4year,0.0,0.0,0.0,0.2,0.0,2.2,0.0,0.2,0.0,0.4,0.2,0.2,0.2
5year,0.1,0.1,0.1,0.2,0.1,2.3,0.1,0.3,0.1,0.4,0.0,0.0,0.0


Pliki z błędem:


,błąd
xgboost_wagi_podsumowanie.csv,ValueError: Nie znaleziono kolumny z etykietą ...
xgboost_wagi_wyniki_foldow.csv,"KeyError: ""W pliku brakuje wskaźników: ['X1', ..."
